# Irrigação inteligente de uma horta comunitária

**Notebook de preparação do trabalho de Aprendizado por Reforço**

Integrantes: [preencher nomes completos]. Professor: [preencher]. Prazo: [preencher].

**Situação:** tema, algoritmos e eventual grupo em dupla aguardam validação do professor. Este notebook prepara a modelagem e o protocolo; não implementa o ambiente nem contém experimentos executados. Registrar a resposta do professor aqui antes da implementação.

Link do vídeo no YouTube: [incluir após gravação]. Link de divulgação nas redes sociais: [opcional].

## 1. Introdução

Uma horta comunitária precisa manter seus canteiros saudáveis com disponibilidade limitada de água. A irrigação envolve decisões sequenciais: irrigar um canteiro agora pode impedir atender outro antes do reabastecimento; esperar pode economizar água quando houver chuva, mas aumentar o risco em períodos secos.

Propomos um simulador didático com quatro canteiros, clima estocástico e reservatório limitado. A contribuição do ambiente será combinar necessidades diferentes dos canteiros, falta e excesso de água e competição pelo reservatório. Trata-se de uma proposta própria para a disciplina; não foi demonstrado ineditismo na literatura nem fidelidade agronômica.

Serão comparados DQN, PPO e A2C, utilizando observações numéricas e ações discretas. **Principais descobertas: pendentes dos experimentos.** Após as execuções, atualizar esta introdução com os resultados efetivamente observados.

## 2. Modelagem do problema

### 2.1 MDP proposto

Definimos o MDP finito como $M=(S,A,P,R,H)$, com fator de desconto $\gamma$ usado no treinamento. As regras abaixo são uma proposta sujeita à validação; fixá-las antes da busca de hiperparâmetros.

| Elemento | Proposta |
|---|---|
| Estado | Umidades $u_1,\ldots,u_4$, saúdes $h_1,\ldots,h_4$, água no reservatório $w$, regime climático $c$ e turnos restantes $\tau$ |
| Observação | Vetor normalizado: umidades, saúdes, fração de água, clima em representação one-hot e fração de tempo restante |
| Ações | Irrigar canteiro 1, 2, 3 ou 4; reabastecer; esperar: seis ações, espaço `Discrete(6)` |
| Transições | Aplicar ação, chuva/evaporação segundo o clima, atualizar saúde, sortear próximo clima e reduzir tempo restante |
| Recompensa | Incentivo à saúde média; custos pelo uso de água e reabastecimento; penalidade pela morte de um canteiro |
| Horizonte | Proposta: 60 turnos de manejo; fim antecipado se todos os canteiros morrerem |
| Inicialização | Distribuições fixas de umidade e água; plantas vivas; clima sorteado usando a semente do ambiente |

O clima será uma cadeia de Markov observável, com regimes seco, ameno e chuvoso. Sua matriz de transição será documentada. A chuva ocorrerá de acordo com o regime atual; a próxima condição dependerá apenas desse regime. As necessidades de cada canteiro serão constantes conhecidas e diferentes entre si.

O tempo restante entra no estado porque a decisão depende da proximidade do fim. A saúde entra porque o efeito de secas anteriores não pode ficar escondido. Esses cuidados evitam que a dinâmica dependa de um histórico ausente da observação.

### 2.2 Recompensa candidata

Uma forma inicial é

$$r_t=\overline{h}_{t+1}-\lambda_w q_t-\lambda_d d_t-\lambda_f\mathbb{1}[a_t=\text{reabastecer}],$$

em que $q_t$ é a água efetivamente usada na irrigação e $d_t$ é o número de plantas que morreram **neste turno**. Os coeficientes, a quantidade por irrigação, a capacidade do reservatório, as faixas de umidade e as taxas de perda/recuperação de saúde estão pendentes de definição. Não usar esta expressão como se já tivesse sido validada.

A morte será irreversível. Uma irrigação sem água não criará água; seu efeito e eventual penalidade serão definidos explicitamente. Reabastecer ocupará um turno, permitindo que a umidade dos outros canteiros mude durante a ação.

### 2.3 Implementação e visualização previstas

Após aprovação, criar uma classe derivada de `gymnasium.Env`, com `reset(seed=...)`, `step`, `render` e `close`. Usar o gerador `self.np_random` e validar o ambiente com `check_env`. O campo `info` fornecerá métricas de água e saúde para a avaliação.

Como o horizonte proposto faz parte do MDP e aparece no estado, seu encerramento é terminal. Um limite externo adicional de execução, se existir, será uma truncagem. Documentar essa distinção.

Renderização proposta: quatro canteiros com barras de umidade e saúde, nível do reservatório, clima, turno e ação. Disponibilizar `rgb_array` para gravar a execução escolhida para o vídeo. A política recebe o vetor numérico, não a imagem renderizada.

## 3. Metodologia de experimentos

### 3.1 Algoritmos e justificativas

| Algoritmo | Característica | Justificativa |
|---|---|---|
| DQN | Aprende valores de ação com rede neural; reutiliza transições em replay buffer | Relaciona o trabalho ao Q-learning tabular estudado e permite avaliar uma abordagem off-policy |
| PPO | Aprende uma política e função de valor; usa objetivo com clipping | Já utilizado nos notebooks da disciplina; fornece uma referência on-policy |
| A2C | Atualiza ator e crítico com estimativas de vantagem | Permite comparar outro método on-policy com atualização diferente da do PPO |

Os três suportam ações discretas. Usaremos `MlpPolicy`, pois o estado é numérico. DQN terá duas camadas ocultas de 64 unidades; PPO/A2C terão redes de ator e crítico com duas camadas de 64 unidades. Isso não iguala o número total de parâmetros nem o custo computacional; comparar também o tempo de treino.

Referências oficiais: [DQN](https://stable-baselines3.readthedocs.io/en/master/modules/dqn.html), [PPO](https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html), [A2C](https://stable-baselines3.readthedocs.io/en/master/modules/a2c.html) e [orientações de experimentação](https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html). A documentação consultada pode mostrar uma versão de desenvolvimento; registrar e fixar a versão estável realmente instalada.

### 3.2 Hiperparâmetros

O arquivo JSON acompanha este notebook e contém uma configuração base e três variações por algoritmo. Cada variação altera um hiperparâmetro em relação à base. Esta busca pequena e controlada atende à comparação das tentativas, mas não constitui otimização exaustiva.

- Comuns: `learning_rate` regula o tamanho das atualizações; `gamma` controla o desconto; `policy_kwargs` define a arquitetura.
- DQN: `buffer_size`, `learning_starts`, `batch_size`, `train_freq`, `gradient_steps`, `target_update_interval` e parâmetros de exploração controlam reutilização de experiências, atualizações e ações aleatórias.
- PPO: `n_steps`, `batch_size`, `n_epochs`, `gae_lambda`, `clip_range` e `ent_coef` controlam coleta, atualização, estimação de vantagem e exploração.
- A2C: `n_steps`, `gae_lambda`, `ent_coef`, `vf_coef`, `max_grad_norm` e `use_rms_prop` controlam coleta, perdas e otimizador.

Os valores abaixo são pontos de partida propostos, **não configurações vencedoras**. Outros parâmetros do construtor usarão os padrões da versão instalada; salvar a versão e os parâmetros efetivos junto a cada execução.

In [ ]:
from pathlib import Path
import json
import copy
import csv
import sys
import importlib.metadata

ROOT = Path.cwd()
if not (ROOT / "configs" / "experimentos.json").exists():
    raise FileNotFoundError("Abra o notebook a partir da pasta do trabalho.")
CONFIG = json.loads((ROOT / "configs" / "experimentos.json").read_text(encoding="utf-8"))

CANDIDATES = {}
for algorithm, specification in CONFIG["algorithms"].items():
    for variant in specification["variants"]:
        parameters = copy.deepcopy(specification["base"])
        parameters.update(variant["changes"])
        CANDIDATES[(algorithm, variant["id"])] = parameters

for (algorithm, variant), parameters in CANDIDATES.items():
    print(algorithm, variant, json.dumps(parameters, ensure_ascii=False))

### 3.3 Protocolo e orçamento

1. **Antes da implementação:** obter validação do professor e confirmar integrantes/prazo.
2. **Piloto após implementar:** verificar API, renderização e viabilidade de treino com as bases. Definir o orçamento a partir do custo observado. Mudanças feitas nessa fase precisam ser registradas antes da busca.
3. **Busca:** quatro configurações por algoritmo, três sementes independentes de treinamento e o mesmo orçamento de interações por execução. Avaliar o modelo ao final em 30 episódios de validação.
4. **Seleção:** escolher por algoritmo a configuração com maior média de retorno de validação, agregada primeiro por semente de treinamento e depois entre sementes. Em empate numérico, preferir menor consumo de água. Não selecionar modelos com o conjunto de teste.
5. **Comparação final:** retreinar do zero cada configuração escolhida com cinco novas sementes e um orçamento comum maior. Avaliar cada modelo em 100 episódios de teste, sem aprendizado e com `deterministic=True`.
6. **Relato:** apresentar todas as configurações, curvas de treino, métricas de validação e teste, variabilidade e tempo. Usar o modelo do fim do orçamento, sem escolher retrospectivamente um checkpoint favorável.

Os orçamentos foram escolhidos como múltiplos dos tamanhos de rollout propostos para reduzir diferenças de coleta: 20.480 passos no piloto, 102.400 na busca e 307.200 no treino final, com um ambiente por execução. Registrar o número real de passos de cada modelo.

As sementes dos episódios de validação e teste são separadas das sementes dos treinamentos e repetidas entre os algoritmos para permitir cenários comparáveis. Os processos aleatórios externos do ambiente deverão avançar de forma independente das ações, para que a mesma semente represente o mesmo clima. Reservar a lista de teste para a etapa final.

**Custo proposto:** 3 pilotos, 36 treinos de busca e 15 treinos finais; 8.355.840 interações de treinamento no total, sem contar avaliação. A duração ainda não foi medida. Se o orçamento precisar mudar após o piloto, alterar igualmente o protocolo e registrar a justificativa antes da busca.

In [ ]:
runs = {
    "pilot": len(CONFIG["algorithms"]) * len(CONFIG["seeds"]["pilot"]),
    "tuning": len(CANDIDATES) * len(CONFIG["seeds"]["tuning"]),
    "final": len(CONFIG["algorithms"]) * len(CONFIG["seeds"]["final"]),
}
total_steps = sum(runs[stage] * CONFIG["budgets"][stage] for stage in runs)
print("Execuções por etapa:", runs)
print("Interações de treinamento previstas:", f"{total_steps:,}")

### 3.4 Métricas e referências de comparação

- **Retorno episódico:** soma não descontada das recompensas; métrica principal da seleção.
- **Sobrevivência final:** fração dos quatro canteiros com saúde maior que zero.
- **Saúde final média:** mede qualidade além da simples sobrevivência.
- **Consumo de água:** soma da água efetivamente aplicada; interpretar junto à saúde, pois não irrigar pode consumir zero e matar as plantas.
- **Sucesso:** proposta inicial: alcançar o horizonte com todos os canteiros vivos e saúde média de pelo menos 0,5. Fixar essa definição antes da busca.
- **Tempo de treinamento e passos reais:** custo computacional e custo em interações.

Comparar também com uma política aleatória e uma regra simples: irrigar o canteiro vivo com maior déficit relativo à sua faixa ideal; se não houver água, reabastecer; se nenhum precisar de água, esperar. Avaliar essas referências no mesmo teste.

Apresentar média e desvio padrão entre as cinco sementes de treinamento para cada métrica. Os 100 episódios de um modelo não equivalem a 100 treinamentos independentes. Para sucesso, reportar a taxa por modelo e sua dispersão entre sementes. Curvas suavizadas poderão ajudar na visualização, mas preservar os dados brutos e indicar a janela de suavização.

### 3.5 Registro e reprodutibilidade

Salvar por execução: algoritmo, configuração completa, semente, etapa, versão do ambiente, versões das dependências, orçamento solicitado e realizado, hardware, duração, modelo final e resultados por episódio. Salvar todas as tentativas, inclusive falhas e configurações ruins, com motivo/status.

A tabela por episódio deve usar as colunas definidas abaixo. `water_used` será medido em unidades sintéticas, sem alegar litros reais. Agregar só execuções concluídas e manter o registro das falhas em uma tabela de execuções separada.

In [ ]:
EPISODE_COLUMNS = [
    "stage", "algorithm", "config_id", "training_seed", "evaluation_seed",
    "episode_return", "episode_length", "survival_fraction",
    "final_mean_health", "water_used", "success",
]
RUN_COLUMNS = [
    "stage", "algorithm", "config_id", "training_seed", "status",
    "requested_steps", "actual_steps", "train_seconds", "parameters_json",
    "environment_version", "dependencies_json", "model_path", "error",
]

versions = {"python": sys.version.split()[0]}
for package in ["gymnasium", "stable-baselines3", "torch", "numpy", "pandas", "matplotlib"]:
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = "não instalado"
print(json.dumps(versions, indent=2, ensure_ascii=False))
print("Colunas da avaliação:", EPISODE_COLUMNS)
print("Nenhum treinamento foi executado neste notebook de preparação.")

## 4. Resultados

**Ainda não há resultados numéricos.** Não atribuir desempenho, convergência ou superioridade a um algoritmo antes de executar o protocolo.

Após os experimentos, integrar as células que carregam os dados reais e produzem:

1. Tabela de todas as 12 configurações: hiperparâmetros, retorno de validação por semente, média, dispersão e custo.
2. Justificativa da configuração escolhida para cada algoritmo.
3. Curvas de aprendizado das configurações finalistas, com variabilidade entre sementes e definição dos eixos.
4. Tabela de teste dos três algoritmos, da política aleatória e da heurística: retorno, sobrevivência, saúde, consumo, sucesso e tempo.
5. Discussão dos compromissos entre saúde e água, falhas típicas e comportamento durante chuva/seca.
6. Uma execução ilustrativa e sua renderização; informar a semente e evitar tratá-la como evidência isolada de desempenho.

| Algoritmo | Configuração escolhida | Retorno no teste | Sobrevivência | Água | Sucesso |
|---|---|---|---|---|---|
| DQN | Pendente | Pendente | Pendente | Pendente | Pendente |
| PPO | Pendente | Pendente | Pendente | Pendente | Pendente |
| A2C | Pendente | Pendente | Pendente | Pendente | Pendente |

Esta tabela sinaliza pendências; não contém dados experimentais.

## 5. Conclusões

Preencher depois dos resultados: qual configuração resolveu melhor o problema segundo as métricas, qual foi a variabilidade, quais dificuldades surgiram e quais decisões se mostraram relevantes.

Limitações já identificadas no desenho: dinâmica sintética sem validação agronômica; observação completa idealizada; busca restrita a quatro configurações por algoritmo; diferença de custo e arquitetura entre famílias de algoritmos. Relatar também limitações efetivamente encontradas na execução.

Possíveis trabalhos futuros: sensores ruidosos, reservatório abastecido por chuva, diferentes espécies e comparação com métodos de controle convencionais. Não transformar esses itens em funcionalidades obrigatórias para esta entrega.

## 6. Apresentação e entrega

Produzir vídeo de até três minutos no YouTube com fala de todos os integrantes. O arquivo `apresentacao/roteiro.md` contém uma divisão provisória de 2min50s. Confirmar número de integrantes e adaptar a distribuição de falas.

Antes de entregar: completar nomes, validações, resultados, conclusões, referências e link do vídeo; executar o notebook inteiro em um ambiente limpo; converter para relatório HTML com `jupyter nbconvert --to html trabalho_horta.ipynb`. Conferir textos, códigos, tabelas e figuras na versão exportada. Seguir eventual formato adicional exigido pela plataforma/professor.

### Referências

- Materiais locais da disciplina: notebooks dos módulos 1 a 4; o módulo 3 usa PPO e avaliação de políticas, e os módulos 1/4 exemplificam um ambiente customizado.
- [Stable-Baselines3 — repositório](https://github.com/DLR-RM/stable-baselines3).
- [Gymnasium — criação de ambiente](https://gymnasium.farama.org/tutorials/gymnasium_basics/environment_creation/).
- Documentação dos algoritmos citada na metodologia.

Adicionar os artigos dos algoritmos e outras fontes efetivamente utilizadas na versão final.